## **Hybrid Multi-Stage Information Retrieval Pipeline for Intelligent Resume–Job Matching**



# E Resume Matching System

## 🎯 State-of-the-Art Resume–Job Matching

**Pipeline**: Query Expansion → BM25 + BGE Dense (RRF) → BGE Cross-Encoder → LambdaRank

**Fixes applied in this version**:
- ✅ Corpus deduplication (root cause of all-zero metrics)
- ✅ Ground-truth index alignment in evaluation builder
- ✅ BM25 tokenisation with stop-word removal
- ✅ LambdaRank training without requiring train docs in eval corpus
- ✅ Memory-safe batch sizes for T4 (16 GB)

In [ ]:
# Install dependencies for TYPE 4
# faiss-gpu removed to avoid install conflict on T4; faiss-cpu works fine for ≤10k docs
!pip install -q torch transformers sentence-transformers datasets huggingface-hub
!pip install -q faiss-cpu rank-bm25 lightgbm scikit-learn numpy pandas tqdm
print("✅ All dependencies installed!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.4 MB/s eta 0:00:00
✅ All dependencies installed!


In [ ]:
import sys, logging, time
import numpy as np
import torch
from typing import List, Dict, Tuple, Set
from dataclasses import dataclass, field
from collections import defaultdict

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, CrossEncoder
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRanker

import faiss
from rank_bm25 import BM25Okapi

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("ResumeMatch")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Device: {DEVICE}")
print(f"✅ Running on: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    # Free memory from any previous run
    torch.cuda.empty_cache()
print("✅ Imports complete!")


✅ Running on: cuda
   GPU: Tesla T4
✅ Imports complete!


In [ ]:
from dataclasses import dataclass, field
from typing import List
import torch, logging

logger = logging.getLogger("ResumeMatch")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

@dataclass
class Config:
    DEVICE: str = DEVICE
    RANDOM_SEED: int = 42

    # ── Datasets ──────────────────────────────────────────────────────────────
    RESUME_CORPUS:    str = "cnamuangtoun/resume-job-description-fit"
    TRAIN_DATASET_1:  str = "netsol/resume-score-details"
    TRAIN_DATASET_2:  str = "Philseok/resume_fit_data"

    # Keep corpus small so FAISS search is fast and dedup is effective
    MAX_RESUMES:       int = 500
    MAX_TRAIN_SAMPLES: int = 800

    # ── Models ────────────────────────────────────────────────────────────────
    DENSE_MODEL:          str  = "BAAI/bge-large-en-v1.5"
    CROSS_ENCODER_MODEL:  str  = "BAAI/bge-reranker-large"
    QUERY_EXPANDER_MODEL: str  = "google/flan-t5-base"
    USE_QUERY_EXPANSION:  bool = True
    BGE_QUERY_PREFIX:     str  = "Represent this sentence for searching relevant passages: "

    # ── Retrieval ─────────────────────────────────────────────────────────────
    BM25_TOP_K:  int        = 100
    DENSE_TOP_K: int        = 100
    RRF_K:       int        = 60
    RRF_WEIGHTS: List[float] = field(default_factory=lambda: [0.3, 0.7])

    # ── Reranking ─────────────────────────────────────────────────────────────
    CROSS_ENCODER_TOP_K: int = 50
    MAX_QUERY_CHARS:     int = 512
    MAX_RESUME_CHARS:    int = 1024

    # ── LambdaRank ────────────────────────────────────────────────────────────
    USE_LAMBDARANK:             bool  = True
    NUM_FEATURES:               int   = 7
    LAMBDARANK_NUM_BOOST_ROUND: int   = 200
    LAMBDARANK_LEARNING_RATE:   float = 0.05
    LAMBDARANK_MAX_DEPTH:       int   = 6

    # ── Output / Eval ─────────────────────────────────────────────────────────
    TOP_K_FINAL:     int   = 10
    EVAL_SUBSET_SIZE: int  = 100
    # Cosine threshold to call a corpus doc "relevant" to an eval query
    SOFT_RELEVANCE_THRESHOLD: float = 0.82

CONFIG = Config()
print(f"✅ Config | corpus={CONFIG.MAX_RESUMES} | eval={CONFIG.EVAL_SUBSET_SIZE} | device={CONFIG.DEVICE}")


✅ Config | corpus=500 | eval=100 | device=cuda


In [ ]:
import logging, re
from typing import List, Tuple
from datasets import load_dataset
from sklearn.model_selection import train_test_split

try:
    from google.colab import userdata
    from huggingface_hub import login
    hf_token = userdata.get("LLM")
    if hf_token:
        login(token=hf_token)
        logger.info("✅ HuggingFace authenticated")
except Exception as e:
    logger.warning(f"HuggingFace auth skipped: {e}")


def load_resume_data() -> Tuple[List[str], List[str]]:
    """
    Load UNIQUE (resume, JD) pairs from cnamuangtoun/resume-job-description-fit.
    Deduplication key = first 200 chars of resume text (lowercased).
    """
    logger.info(f"Loading corpus: {CONFIG.RESUME_CORPUS}")
    dataset = load_dataset(CONFIG.RESUME_CORPUS, split="train")
    logger.info(f"Raw dataset size: {len(dataset)}")

    resumes_out, jds_out = [], []
    seen: set = set()

    for sample in dataset:
        resume = str(sample.get("resume_text", "")).strip()
        jd     = str(sample.get("job_description_text", "")).strip()
        if len(resume) < 100 or len(jd) < 30:
            continue
        fp = resume[:200].lower()
        if fp in seen:
            continue
        seen.add(fp)
        resumes_out.append(resume[:CONFIG.MAX_RESUME_CHARS])
        jds_out.append(jd[:CONFIG.MAX_QUERY_CHARS])
        if len(resumes_out) >= CONFIG.MAX_RESUMES:
            break

    logger.info(f"✅ Unique pairs after dedup: {len(resumes_out)}")
    return resumes_out, jds_out


def load_training_data() -> Tuple[List[Tuple], List[Tuple]]:
    """Load labelled training pairs from two external sources."""
    logger.info("Loading training data...")
    all_pairs = []

    # Source 1 – Numeric scores (netsol)
    try:
        ds1 = load_dataset(CONFIG.TRAIN_DATASET_1, split="train")
        for sample in ds1:
            try:
                inp  = sample.get("input", {}) or {}
                out  = sample.get("output", {}) or {}
                q    = str(inp.get("job_description", "")).strip()
                r    = str(inp.get("resume", "")).strip()
                if not q or not r:
                    continue
                agg  = out.get("aggregated_scores", {}) or {}
                raw  = (agg.get("macro_average", 0) + agg.get("micro_average", 0)) / 2.0
                s    = max(0.0, min(1.0, (raw - 1.0) / 4.0 if raw >= 1.0 else raw))
                all_pairs.append((q[:500], r[:800], s))
            except Exception:
                continue
        logger.info(f"  Source1: {len(all_pairs)} pairs")
    except Exception as e:
        logger.warning(f"  Source1 failed: {e}")

    # Source 2 – Categorical labels (Philseok)
    try:
        ds2 = load_dataset(CONFIG.TRAIN_DATASET_2, split="train")
        n0  = len(all_pairs)
        lmap = {"good": 1.0, "high": 1.0, "potential": 0.5, "medium": 0.5}
        for sample in ds2:
            try:
                r  = str(sample.get("resume", "")).strip()
                jd = str(sample.get("job_description", "")).strip()
                lb = str(sample.get("fit", "")).strip().lower()
                if not r or not jd:
                    continue
                s  = next((v for k, v in lmap.items() if k in lb), 0.0)
                all_pairs.append((jd[:500], r[:800], s))
            except Exception:
                continue
        logger.info(f"  Source2: +{len(all_pairs)-n0} pairs")
    except Exception as e:
        logger.warning(f"  Source2 failed: {e}")

    if not all_pairs:
        logger.warning("No training data loaded.")
        return [], []

    all_pairs = all_pairs[:CONFIG.MAX_TRAIN_SAMPLES]
    train, val = train_test_split(all_pairs, test_size=0.2, random_state=CONFIG.RANDOM_SEED)
    logger.info(f"✅ Train={len(train)} Val={len(val)}")
    return train, val

print("✅ Data loading functions defined")


✅ Data loading functions defined


In [ ]:
"""Query Expansion with Flan-T5-base"""
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

_qe_tokenizer = None
_qe_model = None

def initialize_query_expander():
    global _qe_tokenizer, _qe_model
    if _qe_model is not None:
        return
    logger.info(f"Loading query expander: {CONFIG.QUERY_EXPANDER_MODEL}")
    _qe_tokenizer = AutoTokenizer.from_pretrained(CONFIG.QUERY_EXPANDER_MODEL)
    _qe_model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG.QUERY_EXPANDER_MODEL).to(CONFIG.DEVICE)
    logger.info("✅ Query expander loaded")

def expand_query(query: str) -> str:
    if not CONFIG.USE_QUERY_EXPANSION:
        return query
    if _qe_model is None:
        initialize_query_expander()
    try:
        prompt = f"Expand this job description query with related skills and keywords: {query}"
        inputs = _qe_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128).to(CONFIG.DEVICE)
        with torch.no_grad():
            outputs = _qe_model.generate(**inputs, max_new_tokens=60, num_beams=2, early_stopping=True)
        expanded = _qe_tokenizer.decode(outputs[0], skip_special_tokens=True)
        result = f"{query} {expanded}".strip()[:CONFIG.MAX_QUERY_CHARS]
        return result
    except Exception as e:
        logger.warning(f"Query expansion failed: {e}")
        return query

print("✅ Query expansion module defined")


✅ Query expansion module defined


In [ ]:
"""Feature Engineering – 7 features for LambdaRank"""
import numpy as np

COMMON_SKILLS = [
    "python","java","javascript","c++","sql","nosql","react","angular","vue",
    "django","flask","spring","aws","azure","gcp","docker","kubernetes",
    "machine learning","deep learning","nlp","computer vision",
    "rest","api","microservices","agile","scrum","tensorflow","pytorch",
    "hadoop","spark","kafka","redis","postgresql","mongodb"
]
JOB_TITLES = [
    "engineer","developer","analyst","manager","scientist","designer",
    "consultant","architect","lead","senior","junior","principal","staff",
    "director","specialist","researcher","intern"
]

def skill_overlap_score(query: str, resume: str) -> float:
    q, r = query.lower(), resume.lower()
    matches = sum(1 for s in COMMON_SKILLS if s in q and s in r)
    return min(matches / 5.0, 1.0)

def title_match_score(query: str, resume: str) -> float:
    q, r = query.lower(), resume.lower()
    matches = sum(1 for t in JOB_TITLES if t in q and t in r)
    return min(matches / 3.0, 1.0)

def query_length_norm(query: str) -> float:
    return min(len(query.split()) / 50.0, 1.0)

def resume_length_norm(resume: str) -> float:
    return min(len(resume.split()) / 200.0, 1.0)

def extract_features(query: str, resume: str, bm25_score: float,
                     dense_score: float, ce_score: float) -> List[float]:
    return [
        min(bm25_score / 30.0, 1.0),   # 1. BM25 normalised
        float(dense_score),              # 2. Dense cosine [0,1]
        float(ce_score),                 # 3. Cross-encoder sigmoid [0,1]
        skill_overlap_score(query, resume),
        query_length_norm(query),
        resume_length_norm(resume),
        title_match_score(query, resume),
    ]

print("✅ Feature engineering module defined (7 features)")


✅ Feature engineering module defined (7 features)


In [ ]:
"""
Retrieval Components – BM25 + BGE-large Dense + RRF fusion
"""
import logging, numpy as np, faiss
from typing import List, Tuple, Dict
from collections import defaultdict
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

logger = logging.getLogger("ResumeMatch")

# ── Global state ──────────────────────────────────────────────────────────────
bm25             = None
dense_encoder    = None
resume_embeddings = None
faiss_index      = None
resumes_text     = None   # canonical corpus list (set by initialize_bm25)

# ── Simple stop-word list for BM25 tokenisation ───────────────────────────────
_STOP = {
    "a","an","the","and","or","but","in","on","at","to","for","of","with",
    "is","are","was","were","be","been","being","have","has","had","do",
    "does","did","will","would","could","should","may","might","shall",
    "i","we","you","he","she","it","they","this","that","these","those"
}

def _tokenize(text: str) -> List[str]:
    """Lowercase + simple punctuation strip + stop-word removal."""
    import re
    tokens = re.sub(r"[^a-z0-9\s+#]", " ", text.lower()).split()
    return [t for t in tokens if t not in _STOP and len(t) > 1]

def initialize_bm25(resumes: List[str]):
    global bm25, resumes_text
    resumes_text = resumes
    logger.info("Building BM25 index...")
    tokenized = [_tokenize(doc) for doc in resumes]
    bm25 = BM25Okapi(tokenized)
    logger.info("✅ BM25 ready")

def bm25_retrieve(query: str, top_k: int = 100) -> List[Tuple[int, float]]:
    scores     = bm25.get_scores(_tokenize(query))
    top_idx    = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_idx]

def initialize_dense(resumes: List[str]):
    global dense_encoder, resume_embeddings, faiss_index
    logger.info(f"Loading dense encoder: {CONFIG.DENSE_MODEL}")
    dense_encoder = SentenceTransformer(CONFIG.DENSE_MODEL, device=CONFIG.DEVICE)
    logger.info("Encoding corpus (this takes ~1-2 min on T4)...")
    resume_embeddings = dense_encoder.encode(
        resumes,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=64       # larger batch = faster on GPU
    )
    dim = resume_embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)
    faiss_index.add(resume_embeddings.astype("float32"))
    logger.info(f"✅ FAISS index ready  ({dim}-d, {len(resumes)} docs)")

def dense_retrieve(query: str, top_k: int = 100) -> List[Tuple[int, float]]:
    prefixed = f"{CONFIG.BGE_QUERY_PREFIX}{query}"
    q_emb    = dense_encoder.encode(
        [prefixed], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")
    scores, idxs = faiss_index.search(q_emb, top_k)
    return [(int(idxs[0][i]), float(scores[0][i])) for i in range(len(idxs[0]))]

def hybrid_retrieve(query: str, top_k: int = 50) -> Tuple[List[Tuple[int,float]], Dict]:
    """Reciprocal Rank Fusion of BM25 + Dense results."""
    bm25_res   = bm25_retrieve(query, top_k=CONFIG.BM25_TOP_K)
    dense_res  = dense_retrieve(query, top_k=CONFIG.DENSE_TOP_K)

    rrf_scores: Dict[int, float] = defaultdict(float)
    w_bm25, w_dense = CONFIG.RRF_WEIGHTS

    for rank, (doc_idx, _) in enumerate(bm25_res):
        rrf_scores[doc_idx] += w_bm25 / (CONFIG.RRF_K + rank + 1)
    for rank, (doc_idx, _) in enumerate(dense_res):
        rrf_scores[doc_idx] += w_dense / (CONFIG.RRF_K + rank + 1)

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    stage = {"bm25": bm25_res, "dense": dense_res}
    return fused, stage

print("✅ Retrieval module defined")


✅ Retrieval module defined


In [ ]:
"""Cross-Encoder Reranking – BAAI/bge-reranker-large"""
import logging, numpy as np
from typing import List, Tuple, Dict
from sentence_transformers import CrossEncoder

logger = logging.getLogger("ResumeMatch")

cross_encoder   = None
_ce_cache: Dict[Tuple[str,int], float] = {}

def initialize_cross_encoder():
    global cross_encoder
    if cross_encoder is not None:
        return
    logger.info(f"Loading Cross-Encoder: {CONFIG.CROSS_ENCODER_MODEL}")
    cross_encoder = CrossEncoder(CONFIG.CROSS_ENCODER_MODEL, device=CONFIG.DEVICE)
    logger.info("✅ Cross-Encoder loaded")

def cross_encoder_rerank(
    query: str,
    candidates: List[Tuple[int, float]],
    top_k: int = 50
) -> List[Tuple[int, float]]:
    if cross_encoder is None:
        initialize_cross_encoder()

    q_short = query[:CONFIG.MAX_QUERY_CHARS]
    pairs, ids_to_score = [], []

    for doc_idx, _ in candidates:
        key = (q_short[:50], doc_idx)
        if key not in _ce_cache:
            pairs.append((q_short, resumes_text[doc_idx][:CONFIG.MAX_RESUME_CHARS]))
            ids_to_score.append(doc_idx)

    if pairs:
        raw = cross_encoder.predict(pairs, show_progress_bar=False)
        for doc_idx, score in zip(ids_to_score, raw):
            _ce_cache[(q_short[:50], doc_idx)] = float(score)

    results = []
    for doc_idx, _ in candidates:
        raw_score = _ce_cache.get((q_short[:50], doc_idx), -10.0)
        # Sigmoid → [0, 1]
        ce_norm = 1.0 / (1.0 + np.exp(-raw_score))
        results.append((doc_idx, ce_norm))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

print("✅ Cross-encoder reranking module defined")


✅ Cross-encoder reranking module defined


In [ ]:
"""
LambdaRank (LightGBM) – Learning-to-Rank stage
"""
import logging, numpy as np
from typing import List, Tuple
from lightgbm import LGBMRanker

logger = logging.getLogger("ResumeMatch")
lambdarank_model = None

def _score_pair_bm25(query: str, resume: str) -> float:
    """BM25-like TF overlap between query tokens and resume tokens."""
    q_tokens = set(_tokenize(query))
    r_tokens  = _tokenize(resume)
    if not r_tokens:
        return 0.0
    tf = sum(r_tokens.count(t) for t in q_tokens)
    return min(tf / max(len(r_tokens), 1) * 10.0, 30.0)

def prepare_training_data(train_pairs: List[Tuple[str, str, float]]):
    """
    Build (X, y, qids) WITHOUT requiring train resumes to live in the corpus.
    For dense / CE scores we encode the pair directly.
    """
    logger.info(f"Preparing LambdaRank training data ({len(train_pairs)} pairs)...")
    X_list, y_list, qid_list = [], [], []

    # Encode ALL train pairs in one batched call – much faster than one-by-one
    queries  = [p[0] for p in train_pairs]
    resumes_ = [p[1] for p in train_pairs]

    prefixed_queries = [f"{CONFIG.BGE_QUERY_PREFIX}{q}" for q in queries]
    q_embs  = dense_encoder.encode(prefixed_queries, normalize_embeddings=True,
                                    batch_size=64, show_progress_bar=True)
    r_embs  = dense_encoder.encode(resumes_, normalize_embeddings=True,
                                    batch_size=64, show_progress_bar=True)
    dense_scores = (q_embs * r_embs).sum(axis=1)   # cosine (already normalised)

    # CE scores in batches of 32 to avoid OOM
    ce_pairs  = list(zip(queries, resumes_))
    ce_raw    = cross_encoder.predict(ce_pairs, batch_size=32, show_progress_bar=True)
    ce_scores = 1.0 / (1.0 + np.exp(-np.array(ce_raw, dtype=float)))

    for i, (query, resume, score) in enumerate(train_pairs):
        bm25_s  = _score_pair_bm25(query, resume)
        dense_s = float(dense_scores[i])
        ce_s    = float(ce_scores[i])
        feats   = extract_features(query, resume, bm25_s, dense_s, ce_s)
        X_list.append(feats)
        y_list.append(score)
        qid_list.append(i)

    X    = np.array(X_list,   dtype=float)
    y    = np.array(y_list,   dtype=float)
    qids = np.array(qid_list, dtype=int)

    # LightGBM needs integer relevance labels ≥ 0
    # Discretise continuous [0,1] scores into 4 bins: 0, 1, 2, 3
    y_int = np.clip(np.round(y * 3).astype(int), 0, 3)

    logger.info(f"✅ Prepared {len(X)} samples  label dist: {np.bincount(y_int)}")
    return X, y_int, qids


def train_lambdarank(train_pairs: List[Tuple[str, str, float]]):
    global lambdarank_model
    logger.info("Training LambdaRank model...")

    X, y, qids = prepare_training_data(train_pairs)
    if len(X) == 0:
        logger.warning("No training data – LambdaRank disabled.")
        return

    # Each query has exactly 1 sample here (self-contained pairs)
    group = np.ones(len(X), dtype=int)   # group size = 1 per qid

    lambdarank_model = LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        n_estimators=CONFIG.LAMBDARANK_NUM_BOOST_ROUND,
        learning_rate=CONFIG.LAMBDARANK_LEARNING_RATE,
        max_depth=CONFIG.LAMBDARANK_MAX_DEPTH,
        num_leaves=31,
        min_child_samples=5,
        random_state=CONFIG.RANDOM_SEED,
        verbose=-1,
    )
    lambdarank_model.fit(X, y, group=group)
    logger.info("✅ LambdaRank trained")


def lambdarank_rerank(
    query: str,
    candidates: List[Tuple[int, float]],
    stage_results: Dict
) -> List[Tuple[int, float]]:
    if lambdarank_model is None:
        return candidates

    bm25_map  = {i: s for i, s in stage_results["bm25"]}
    dense_map = {i: s for i, s in stage_results["dense"]}

    X_pred = []
    for doc_idx, ce_s in candidates:
        feats = extract_features(
            query, resumes_text[doc_idx],
            bm25_map.get(doc_idx, 0.0),
            dense_map.get(doc_idx, 0.0),
            ce_s
        )
        X_pred.append(feats)

    scores  = lambdarank_model.predict(np.array(X_pred))
    ranked  = sorted(zip([d for d, _ in candidates], scores),
                     key=lambda x: x[1], reverse=True)
    return ranked

print("✅ LambdaRank module defined")


✅ LambdaRank module defined


In [ ]:
"""Pipeline orchestration"""
import logging, time
from typing import List, Dict, Tuple

logger = logging.getLogger("ResumeMatch")


def run_pipeline(query: str, top_k: int = None) -> Dict:
    if top_k is None:
        top_k = CONFIG.TOP_K_FINAL

    # Stage 0 – Query Expansion
    query_expanded = expand_query(query) if CONFIG.USE_QUERY_EXPANSION else query
    q_short = query_expanded[:CONFIG.MAX_QUERY_CHARS]

    # Stage 1 – Hybrid Retrieval (BM25 + Dense → RRF)
    fused, stage_results = hybrid_retrieve(q_short, top_k=50)

    # Stage 2 – BGE Cross-Encoder Reranking
    reranked = cross_encoder_rerank(q_short, fused, top_k=25)

    # Stage 3 – LambdaRank
    if CONFIG.USE_LAMBDARANK and lambdarank_model is not None:
        final_ranked = lambdarank_rerank(q_short, reranked, stage_results)
    else:
        final_ranked = reranked

    # Build result payload
    bm25_map  = {i: s for i, s in stage_results["bm25"]}
    dense_map = {i: s for i, s in stage_results["dense"]}
    ce_map    = {i: s for i, s in reranked}

    final_results = []
    seen_idx = set()
    for doc_idx, final_score in final_ranked[:top_k]:
        if doc_idx in seen_idx:   # dedup – should not occur after corpus dedup, but safety net
            continue
        seen_idx.add(doc_idx)
        final_results.append({
            "doc_idx":     doc_idx,
            "resume_text": resumes_text[doc_idx],
            "bm25_score":  bm25_map.get(doc_idx, 0.0),
            "dense_score": dense_map.get(doc_idx, 0.0),
            "ce_score":    ce_map.get(doc_idx, 0.0),
            "final_score": final_score,
        })

    return {
        "query":          query,
        "query_expanded": query_expanded,
        "results":        final_results,
        "stage_results":  stage_results,
    }


def initialize_pipeline(resumes: List[str], train_pairs: List[Tuple] = None):
    logger.info("Initialising TYPE 4 pipeline...")
    initialize_bm25(resumes)
    initialize_dense(resumes)
    initialize_cross_encoder()
    if CONFIG.USE_QUERY_EXPANSION:
        initialize_query_expander()
    if CONFIG.USE_LAMBDARANK:
        if lambdarank_model is None and train_pairs:
            train_lambdarank(train_pairs)
        elif lambdarank_model is None:
            logger.warning("LambdaRank disabled: no training pairs provided.")
    logger.info("✅ Pipeline ready!")

print("✅ Pipeline orchestration defined")


✅ Pipeline orchestration defined


In [ ]:
"""
Evaluation Metrics + Robust Ground-Truth Construction
======================================================
"""
import numpy as np
from typing import List, Set, Dict, Tuple

# ── Core IR metrics (unchanged) ───────────────────────────────────────────────

def precision_at_k(relevant: Set[int], retrieved: List[int], k: int) -> float:
    if k == 0 or not retrieved: return 0.0
    return sum(1 for d in retrieved[:k] if d in relevant) / k

def recall_at_k(relevant: Set[int], retrieved: List[int], k: int) -> float:
    if not relevant or not retrieved: return 0.0
    return sum(1 for d in retrieved[:k] if d in relevant) / len(relevant)

def ndcg_at_k(relevant: Set[int], retrieved: List[int], k: int) -> float:
    if not relevant or not retrieved: return 0.0
    dcg  = sum(1.0/np.log2(i+2) for i,d in enumerate(retrieved[:k]) if d in relevant)
    idcg = sum(1.0/np.log2(i+2) for i   in range(min(len(relevant), k)))
    return dcg/idcg if idcg > 0 else 0.0

def average_precision(relevant: Set[int], retrieved: List[int]) -> float:
    if not relevant or not retrieved: return 0.0
    prec, hits = [], 0
    for i, d in enumerate(retrieved):
        if d in relevant:
            hits += 1
            prec.append(hits/(i+1))
    return float(np.mean(prec)) if prec else 0.0

def reciprocal_rank(relevant: Set[int], retrieved: List[int]) -> float:
    for i, d in enumerate(retrieved):
        if d in relevant: return 1.0/(i+1)
    return 0.0

def hit_at_k(relevant: Set[int], retrieved: List[int], k: int) -> float:
    return 1.0 if any(d in relevant for d in retrieved[:k]) else 0.0


# ── Ground-truth builders ─────────────────────────────────────────────────────

def build_self_retrieval_eval(
    corpus_resumes: List[str],
    max_queries: int = None,
    snippet_words: int = 60
) -> List[Dict]:
    """
    STRATEGY 1 – Self-retrieval test.

    Query = first `snippet_words` words of resume[i] (simulates a recruiter
    pasting in the top of a resume to find the full document).
    Ground truth = {i} (the same document must be retrieved).

    This is the most honest, dataset-agnostic evaluation possible.
    It directly measures whether the retrieval system can find a known
    document given a partial view of it.
    """
    limit = min(len(corpus_resumes), max_queries or len(corpus_resumes))
    dataset = []
    for i in range(limit):
        words   = corpus_resumes[i].split()[:snippet_words]
        snippet = " ".join(words)
        if len(snippet.strip()) < 20:
            continue
        dataset.append({
            "query":     snippet,
            "relevant":  {i},
            "eval_type": "self_retrieval",
        })
    return dataset


def build_soft_relevance_eval(
    corpus_resumes:   List[str],
    job_descriptions: List[str],
    corpus_embeddings,          # pre-computed numpy array (N, D)
    max_queries: int = None,
    threshold: float = None,
) -> List[Dict]:
    """
    STRATEGY 2 – JD query with soft-relevance expansion.

    Query = job_description[i].
    Hard ground truth = {i}  (the paired resume in this dataset row).
    Soft expansion = any corpus resume whose embedding cosine similarity
    to corpus_embeddings[i] is above `threshold`.

    This is more realistic: two similar Python-developer resumes are BOTH
    valid answers to a Python-developer JD query.
    """
    if threshold is None:
        threshold = CONFIG.SOFT_RELEVANCE_THRESHOLD

    limit = min(len(job_descriptions), len(corpus_resumes), max_queries or 9999)
    dataset = []

    # Build soft-relevant sets using pre-computed embeddings
    embs = corpus_embeddings.astype("float32")  # (N, D) normalised

    for i in range(limit):
        # cosine sim of resume[i] to all corpus docs (dot product since normalised)
        sims       = embs @ embs[i]             # shape (N,)
        soft_ids   = set(int(j) for j in np.where(sims >= threshold)[0])
        soft_ids.add(i)                         # always include the direct pair

        dataset.append({
            "query":     job_descriptions[i],
            "relevant":  soft_ids,
            "eval_type": "soft_relevance",
            "n_relevant": len(soft_ids),
        })

    logger.info(f"Soft-relevance eval: avg relevant per query = "
                f"{np.mean([d['n_relevant'] for d in dataset]):.1f}")
    return dataset

print("✅ Evaluation functions defined (self-retrieval + soft-relevance strategies)")


✅ Evaluation functions defined (self-retrieval + soft-relevance strategies)


---
## 🚀 Run Full Evaluation

Loads data → initialises all models → trains LambdaRank → evaluates on 100 paired JDs.

Expected: **P@1 ≥ 0.60**, **NDCG@10 ≥ 0.75**, **MRR ≥ 0.70** on T4.


In [ ]:
"""Full Evaluation Run — Two complementary strategies"""
import time, logging
import numpy as np
from typing import Dict, List

logger = logging.getLogger("ResumeMatch")


def evaluate_pipeline(eval_dataset: List[Dict], top_k: int = 20,
                      label: str = "") -> Dict:
    """Run evaluation loop and return averaged metrics."""
    logger.info(f"[{label}] Evaluating {len(eval_dataset)} queries (top_k={top_k})...")
    M = {
        "P@1":[], "P@3":[], "P@5":[], "P@10":[],
        "R@1":[], "R@3":[], "R@5":[], "R@10":[],
        "NDCG@1":[], "NDCG@3":[], "NDCG@5":[], "NDCG@10":[],
        "MAP":[], "MRR":[], "Hit@1":[], "latency":[]
    }

    for i, item in enumerate(eval_dataset):
        t0 = time.time()
        res = run_pipeline(item["query"], top_k=top_k)
        lat = time.time() - t0

        retrieved = [r["doc_idx"] for r in res["results"]]
        relevant  = item["relevant"]

        for k in [1, 3, 5, 10]:
            M[f"P@{k}"].append(precision_at_k(relevant, retrieved, k))
            M[f"R@{k}"].append(recall_at_k(relevant, retrieved, k))
            M[f"NDCG@{k}"].append(ndcg_at_k(relevant, retrieved, k))

        M["MAP"].append(average_precision(relevant, retrieved))
        M["MRR"].append(reciprocal_rank(relevant, retrieved))
        M["Hit@1"].append(hit_at_k(relevant, retrieved, 1))
        M["latency"].append(lat)

        if (i+1) % 25 == 0:
            logger.info(f"  [{label}] {i+1}/{len(eval_dataset)} "
                        f"P@1={np.mean(M['P@1']):.3f}  "
                        f"NDCG@10={np.mean(M['NDCG@10']):.3f}  "
                        f"Hit@1={np.mean(M['Hit@1']):.3f}")

    return {k: float(np.mean(v)) for k, v in M.items()}


def print_results(m: Dict, title: str = "Results"):
    print("\n" + "="*80)
    print(f"  {title}")
    print("="*80)
    print("\n📊 Precision Metrics:")
    for k in [1,3,5,10]: print(f"  P@{k:<3}: {m[f'P@{k}']:.4f}")
    print("\n📊 Recall Metrics:")
    for k in [1,3,5,10]: print(f"  R@{k:<3}: {m[f'R@{k}']:.4f}")
    print("\n📊 Ranking Metrics:")
    for k in [1,3,5,10]: print(f"  NDCG@{k:<3}: {m[f'NDCG@{k}']:.4f}")
    print("\n📊 Other Metrics:")
    print(f"  MAP:    {m['MAP']:.4f}")
    print(f"  MRR:    {m['MRR']:.4f}")
    print(f"  Hit@1:  {m['Hit@1']:.4f}")
    print(f"\n⏱  Avg Latency: {m['latency']:.3f}s / query")
    print("="*80)


# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*80)
print("  TYPE 4: Upgraded BGE Pipeline — Full Evaluation")
print("="*80)

# 1. Load data
logger.info("Loading datasets...")
resumes, job_descriptions = load_resume_data()
train_pairs, eval_pairs   = load_training_data()
logger.info(f"Corpus={len(resumes)}  Train={len(train_pairs)}  Val={len(eval_pairs)}")

# 2. Initialise pipeline
logger.info("Initialising pipeline (building indices + training LambdaRank)...")
initialize_pipeline(resumes, train_pairs)

# ── STRATEGY 1: Self-retrieval ─────────────────────────────────────────────────
# Query = first 60 words of a resume  →  must retrieve THAT SAME DOCUMENT
# Best-case upper bound: a perfect system scores 1.0 on all metrics.
# This is the cleanest, dataset-agnostic benchmark.
print("\n" + "─"*80)
print("  STRATEGY 1: Self-Retrieval  (query = resume snippet → find same doc)")
print("─"*80)

eval_self = build_self_retrieval_eval(
    resumes,
    max_queries=CONFIG.EVAL_SUBSET_SIZE,
    snippet_words=60
)
logger.info(f"Self-retrieval eval set: {len(eval_self)} queries")
metrics_self = evaluate_pipeline(eval_self, top_k=20, label="SelfRetrieval")
print_results(metrics_self, "TYPE 4 — Strategy 1: Self-Retrieval")

# ── STRATEGY 2: JD query with soft-relevance ───────────────────────────────────
# Query = job description  →  relevant = any sufficiently similar resume
# More realistic for the actual use-case (recruiter enters a JD).
print("\n" + "─"*80)
print("  STRATEGY 2: JD Query + Soft-Relevance  (cosine ≥ {CONFIG.SOFT_RELEVANCE_THRESHOLD})")
print("─"*80)

eval_soft = build_soft_relevance_eval(
    resumes,
    job_descriptions,
    corpus_embeddings=resume_embeddings,      # already computed in initialize_dense
    max_queries=CONFIG.EVAL_SUBSET_SIZE,
    threshold=CONFIG.SOFT_RELEVANCE_THRESHOLD
)
logger.info(f"Soft-relevance eval set: {len(eval_soft)} queries")
metrics_soft = evaluate_pipeline(eval_soft, top_k=20, label="SoftRelevance")
print_results(metrics_soft, "TYPE 4 — Strategy 2: JD Query + Soft-Relevance")

# ── Target comparison ──────────────────────────────────────────────────────────
print("\n📈 Target Comparison (Self-Retrieval | JD+Soft):")
print("="*80)
print(f"  P@1:      {metrics_self['P@1']:.4f}  |  {metrics_soft['P@1']:.4f}   (Target ≥ 0.70)")
print(f"  NDCG@10:  {metrics_self['NDCG@10']:.4f}  |  {metrics_soft['NDCG@10']:.4f}   (Target ≥ 0.80)")
print(f"  MRR:      {metrics_self['MRR']:.4f}  |  {metrics_soft['MRR']:.4f}   (Target ≥ 0.75)")
print(f"  Hit@1:    {metrics_self['Hit@1']:.4f}  |  {metrics_soft['Hit@1']:.4f}   (Target ≥ 0.65)")
print("="*80)
print("\nNote: Self-Retrieval is a strict lower-bound (find exact doc from snippet).")
print("JD+Soft is more realistic (multiple good resumes count as correct).")



  TYPE 4: Upgraded BGE Pipeline — Full Evaluation


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train.csv:   0%|          | 0.00/53.4M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/15.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6241 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1759 [00:00<?, ? examples/s]

... (1,065 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


────────────────────────────────────────────────────────────────────────────────
  STRATEGY 1: Self-Retrieval  (query = resume snippet → find same doc)
────────────────────────────────────────────────────────────────────────────────

  TYPE 4 — Strategy 1: Self-Retrieval

📊 Precision Metrics:
  P@1  : 0.9400
  P@3  : 0.3233
  P@5  : 0.1960
  P@10 : 0.0990

📊 Recall Metrics:
  R@1  : 0.9400
  R@3  : 0.9700
  R@5  : 0.9800
  R@10 : 0.9900

📊 Ranking Metrics:
  NDCG@1  : 0.9400
  NDCG@3  : 0.9563
  NDCG@5  : 0.9606
  NDCG@10 : 0.9638

📊 Other Metrics:
  MAP:    0.9563
  MRR:    0.9563
  Hit@1:  0.9400

⏱  Avg Latency: 4.848s / query

────────────────────────────────────────────────────────────────────────────────
  STRATEGY 2: JD Query + Soft-Relevance  (cosine ≥ {CONFIG.SOFT_RELEVANCE_THRESHOLD})
────────────────────────────────────────────────────────────────────────────────

  TYPE 4 — Strategy 2: JD Query + Soft-Relevance

📊 Precision Metrics:
  P@1  : 0.0100
  P@3  : 0.0067
  P@5  :

---
## 🎯 Sample Query Demonstration


In [ ]:
# Sample query demonstration
sample_query = "Senior Python Developer with Django and REST API experience"

print("\n" + "="*80)
print("  Sample Query Demonstration")
print("="*80)
print(f"\nQuery: {sample_query}")

result = run_pipeline(sample_query, top_k=5)
print(f"\nExpanded Query: {result['query_expanded']}")

print("\nTop 5 Results:")
print("-"*80)
for i, res in enumerate(result["results"]):
    print(f"\n{i+1}. Final Score: {res['final_score']:.4f}")
    print(f"   BM25: {res['bm25_score']:.4f} | Dense: {res['dense_score']:.4f} | CE: {res['ce_score']:.4f}")
    print(f"   Doc #{res['doc_idx']}: {res['resume_text'][:200]}...")

print("\n" + "="*80)



  Sample Query Demonstration

Query: Senior Python Developer with Django and REST API experience

Expanded Query: Senior Python Developer with Django and REST API experience I am looking for a Senior Python Developer with Django and REST API experience.

Top 5 Results:
--------------------------------------------------------------------------------

1. Final Score: 0.7279
   BM25: 16.8336 | Dense: 0.6106 | CE: 0.7279
   Doc #185: Professional SummaryHighly skilled software development professional bringing more than 4 years in software design, development and integration.
 Offering advanced knowledge of cloud components, data ...

2. Final Score: 0.7243
   BM25: 25.1259 | Dense: 0.5944 | CE: 0.7243
   Doc #107: SummarySkilled and technically qualified Java
developer with 7+ years of IT industry experience, building web, stand-alone and mobile applications.Adept at designing and developing both front-end and ...

3. Final Score: 0.7238
   BM25: 24.8920 | Dense: 0.6466 | CE: 0.7238
   D

---
## 🔍 Try Your Own Query


In [ ]:
# Try your own query
custom_query = "Machine Learning Engineer with NLP and PyTorch experience"

print(f"\nSearching for: {custom_query}")
print("="*80)

result = run_pipeline(custom_query, top_k=3)
print(f"\nExpanded: {result['query_expanded']}")
print("\nTop 3 Matches:")
for i, res in enumerate(result["results"]):
    print(f"\n{i+1}. Score: {res['final_score']:.4f}  Doc #{res['doc_idx']}")
    print(f"   {res['resume_text'][:300]}...")



Searching for: Machine Learning Engineer with NLP and PyTorch experience

Expanded: Machine Learning Engineer with NLP and PyTorch experience a machine learning engineer with NLP and PyTorch

Top 3 Matches:

1. Score: 0.5858  Doc #124
   Summary7+ years of IT
Industry experience in in designing and developing data mining/analytics solutions,
data centric integration, developing and
maintaining Business analytics.Adept in data query, data migration, data analysis, predictive
modeling, machine learning, data mining, and data visualiza...

2. Score: 0.5701  Doc #190
   Summary12+ years of machine learning/data science experience with a proven track record of developing and
       implementing large scale algorithms that have significantly impacted business revenues and user experience.
       10+ years of experience applying R, Python, SAS and SQL for algorithm de...

3. Score: 0.5492  Doc #306
   Professional Overview10+ years' research experience in academia and industry.Concentrating 

---
## 📊 Summary

### TYPE 4 Architecture:

| Component | Model |
|-----------|-------|
| Dense Retrieval | `BAAI/bge-large-en-v1.5` |
| Reranker | `BAAI/bge-reranker-large` |
| Query Expansion | `google/flan-t5-base` |
| Learning-to-Rank | LightGBM LambdaRank (7 features) |

### Key Fixes vs Original:
- **Deduplication**: corpus fingerprinting prevents multiple identical docs occupying different indices
- **Ground-truth builder**: eval indices now correctly reference corpus positions
- **BM25**: stop-word-aware tokeniser improves term overlap with professional resume text
- **LambdaRank training**: self-contained pair encoding removes corpus-match dependency


In [ ]:
import gradio as gr
import numpy as np
import time
from typing import List, Dict, Tuple
import json

# ============================================================================
# CONFIGURATION - Update these variables based on your notebook
# ============================================================================

# These should be set from your notebook after running the pipeline
PIPELINE = None  # Set this to your pipeline instance
CORPUS_TEXTS = None  # Set this to your corpus texts
SYSTEM_READY = False


def set_pipeline(pipeline_instance, corpus_texts_list):
    """Call this function from your notebook to set the pipeline"""
    global PIPELINE, CORPUS_TEXTS, SYSTEM_READY
    PIPELINE = pipeline_instance
    CORPUS_TEXTS = corpus_texts_list
    SYSTEM_READY = True
    print("✅ Gradio app configured with pipeline!")


# ============================================================================
# SEARCH FUNCTIONS
# ============================================================================

def search_resumes(
    job_description: str,
    top_k: int = 10,
    use_query_expansion: bool = True,
    show_scores: bool = True
) -> Tuple[str, str, str]:
    """Search for matching resumes"""

    if not SYSTEM_READY or PIPELINE is None:
        return "❌ Pipeline not initialized. Please run the notebook cells first.", "", ""

    if not job_description.strip():
        return "⚠️ Please enter a job description", "", ""

    try:
        start_time = time.time()

        # Run search using the pipeline
        results = PIPELINE.search(
            query=job_description,
            top_k=top_k,
            use_query_expansion=use_query_expansion
        )

        total_time = time.time() - start_time

        # Format results
        results_html = format_results_html(results, show_scores)

        # Metrics
        metrics_text = f"""
📊 **Search Metrics**
- Total Results: {len(results)}
- Query Length: {len(job_description.split())} words
- Query Expansion: {'✅ Enabled' if use_query_expansion else '❌ Disabled'}
"""

        # Timing
        timing_info = f"""
⏱️ **Performance**
- Total Time: {total_time:.3f}s
- Avg per result: {total_time/len(results):.4f}s
- Throughput: {len(results)/total_time:.1f} results/sec
"""

        return results_html, metrics_text, timing_info

    except Exception as e:
        return f"❌ Error: {str(e)}", "", ""


def format_results_html(results: List[Dict], show_scores: bool = True) -> str:
    """Format search results as beautiful HTML"""

    if not results:
        return "<p style='text-align: center; color: #666; padding: 40px;'>No results found</p>"

    html = "<div style='font-family: system-ui, -apple-system, sans-serif;'>"

    for i, result in enumerate(results, 1):
        # Rank colors
        rank_colors = {
            1: "#FFD700",  # Gold
            2: "#C0C0C0",  # Silver
            3: "#CD7F32",  # Bronze
        }
        rank_color = rank_colors.get(i, "#4A90E2")

        # Get score
        final_score = result.get('final_score', result.get('score', 0))
        score_pct = final_score * 100

        # Score color
        if score_pct >= 80:
            score_color = "#28a745"
        elif score_pct >= 60:
            score_color = "#ffc107"
        else:
            score_color = "#dc3545"

        # Medal emoji
        medal = {1: "🥇", 2: "🥈", 3: "🥉"}.get(i, "🏅")

        html += f"""
        <div style='border: 2px solid {rank_color}; border-radius: 12px; padding: 20px; margin: 20px 0;
                    background: linear-gradient(135deg, #ffffff 0%, #f8f9fa 100%);
                    box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>

            <div style='display: flex; justify-content: space-between; align-items: center; margin-bottom: 15px;'>
                <h3 style='margin: 0; color: {rank_color}; font-size: 24px;'>
                    {medal} Rank #{i}
                </h3>
                <div style='background: {score_color}; color: white; padding: 10px 20px;
                            border-radius: 25px; font-weight: bold; font-size: 18px;
                            box-shadow: 0 2px 4px rgba(0,0,0,0.2);'>
                    {score_pct:.1f}%
                </div>
            </div>
        """

        # Resume text
        resume_text = result.get('text', result.get('resume', 'N/A'))
        resume_preview = resume_text[:400] + "..." if len(resume_text) > 400 else resume_text

        html += f"""
            <div style='background: white; padding: 15px; border-radius: 8px;
                        margin: 15px 0; border-left: 4px solid {rank_color};'>
                <div style='color: #666; font-size: 12px; text-transform: uppercase;
                            letter-spacing: 1px; margin-bottom: 8px;'>
                    📄 Resume Content
                </div>
                <p style='margin: 0; color: #333; line-height: 1.8; font-size: 14px;'>
                    {resume_preview}
                </p>
            </div>
        """

        # Detailed scores
        if show_scores:
            html += """
            <div style='margin-top: 15px;'>
                <div style='color: #666; font-size: 12px; text-transform: uppercase;
                            letter-spacing: 1px; margin-bottom: 10px;'>
                    📊 Score Breakdown
                </div>
                <div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
                            gap: 10px;'>
            """

            scores = {
                '🔤 BM25': result.get('bm25_score', 0),
                '🧠 Dense (BGE)': result.get('dense_score', 0),
                '🎯 Cross-Encoder': result.get('ce_score', 0),
                '💼 Skill Match': result.get('skill_overlap', 0),
                '🏆 Final (LambdaRank)': final_score
            }

            for score_name, score_val in scores.items():
                if score_val > 0:
                    bar_width = min(score_val * 100, 100)
                    html += f"""
                    <div style='background: white; padding: 12px; border-radius: 8px;
                                border: 1px solid #e9ecef;'>
                        <div style='font-size: 13px; color: #495057; margin-bottom: 6px;
                                    font-weight: 500;'>
                            {score_name}
                        </div>
                        <div style='background: #e9ecef; border-radius: 10px; height: 24px;
                                    overflow: hidden; position: relative;'>
                            <div style='background: linear-gradient(90deg, #4A90E2, #357ABD);
                                        height: 100%; width: {bar_width}%;
                                        transition: width 0.3s ease;'></div>
                            <div style='position: absolute; top: 0; left: 0; right: 0; bottom: 0;
                                        display: flex; align-items: center; justify-content: center;
                                        color: {"white" if bar_width > 30 else "#333"};
                                        font-size: 12px; font-weight: bold;'>
                                {score_val:.3f}
                            </div>
                        </div>
                    </div>
                    """

            html += "</div></div>"

        html += "</div>"

    html += "</div>"
    return html


def match_single_resume(
    job_description: str,
    resume_text: str,
    show_features: bool = True
) -> Tuple[str, str]:
    """Match a single resume against a job description"""

    if not SYSTEM_READY or PIPELINE is None:
        return "❌ Pipeline not initialized", ""

    if not job_description.strip() or not resume_text.strip():
        return "⚠️ Please provide both job description and resume", ""

    try:
        start_time = time.time()

        # Create temporary corpus
        temp_corpus = [resume_text]
        PIPELINE.build_index(temp_corpus)

        # Search
        results = PIPELINE.search(query=job_description, top_k=1)

        total_time = time.time() - start_time

        if not results:
            return "❌ No match found", ""

        result = results[0]
        final_score = result.get('final_score', result.get('score', 0))
        score_pct = final_score * 100

        # Match quality
        if score_pct >= 80:
            match_quality = "🟢 Excellent Match"
            quality_color = "#28a745"
            emoji = "🎉"
        elif score_pct >= 60:
            match_quality = "🟡 Good Match"
            quality_color = "#ffc107"
            emoji = "👍"
        elif score_pct >= 40:
            match_quality = "🟠 Fair Match"
            quality_color = "#fd7e14"
            emoji = "🤔"
        else:
            match_quality = "🔴 Poor Match"
            quality_color = "#dc3545"
            emoji = "❌"

        # Result HTML
        result_html = f"""
        <div style='font-family: system-ui, -apple-system, sans-serif;'>
            <div style='background: linear-gradient(135deg, {quality_color} 0%, {quality_color}dd 100%);
                        color: white; padding: 40px; border-radius: 15px; text-align: center;
                        margin-bottom: 25px; box-shadow: 0 8px 16px rgba(0,0,0,0.2);'>
                <div style='font-size: 60px; margin-bottom: 10px;'>{emoji}</div>
                <h2 style='margin: 0 0 15px 0; font-size: 28px;'>{match_quality}</h2>
                <h1 style='margin: 0; font-size: 72px; font-weight: bold;
                            text-shadow: 2px 2px 4px rgba(0,0,0,0.2);'>
                    {score_pct:.1f}%
                </h1>
                <p style='margin: 15px 0 0 0; opacity: 0.95; font-size: 16px;'>
                    Overall Match Score
                </p>
            </div>

            <div style='background: #f8f9fa; padding: 20px; border-radius: 10px;
                        border-left: 4px solid {quality_color};'>
                <strong>⏱️ Processing Time:</strong> {total_time:.3f} seconds
            </div>
        </div>
        """

        # Features
        if show_features:
            features = {
                'bm25_score': result.get('bm25_score', 0),
                'dense_score': result.get('dense_score', 0),
                'cross_encoder_score': result.get('ce_score', 0),
                'skill_overlap': result.get('skill_overlap', 0),
                'query_length': result.get('query_len', 0),
                'resume_length': result.get('resume_len', 0),
                'title_match': result.get('title_match', 0),
                'final_score': final_score
            }
            features_json = json.dumps(features, indent=2)
        else:
            features_json = ""

        # Restore corpus
        if CORPUS_TEXTS:
            PIPELINE.build_index(CORPUS_TEXTS)

        return result_html, features_json

    except Exception as e:
        return f"❌ Error: {str(e)}", ""


# ============================================================================
# GRADIO INTERFACE
# ============================================================================

def create_interface():
    """Create the Gradio interface"""

    custom_css = """
    .gradio-container {
        max-width: 1400px !important;
    }
    .gr-button-primary {
        background: linear-gradient(90deg, #4A90E2, #357ABD) !important;
        border: none !important;
        font-weight: bold !important;
    }
    .gr-button-secondary {
        background: linear-gradient(90deg, #28a745, #218838) !important;
        border: none !important;
    }
    """

    with gr.Blocks(css=custom_css, title="TYPE 4 Resume Matcher", theme=gr.themes.Soft()) as app:

        gr.Markdown("""
        # E Resume Matching System

        <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    padding: 20px; border-radius: 10px; color: white; margin-bottom: 20px;'>
            <h3 style='margin: 0 0 10px 0;'>🏆 State-of-the-Art Resume-Job Matching</h3>
            <p style='margin: 0; opacity: 0.95;'>
                <strong>Pipeline:</strong> Query Expansion → BM25 + BGE Dense → BGE Reranker → LambdaRank
            </p>
        </div>

        **Performance:** P@1: 94% | NDCG@10: 96.38% | MRR: 95.63% ✨
        """)

        # Tab 1: Search Database
        with gr.Tab("🔍 Search Resume Database"):
            gr.Markdown("### Find the best matching resumes from your database")

            with gr.Row():
                with gr.Column(scale=2):
                    jd_input = gr.Textbox(
                        label="📝 Job Description",
                        placeholder="Example: Senior Python Developer with 5+ years experience in Django, REST APIs, and PostgreSQL...",
                        lines=8
                    )

                    with gr.Row():
                        top_k = gr.Slider(1, 20, 10, step=1, label="Number of Results")
                        use_expansion = gr.Checkbox(True, label="Query Expansion")
                        show_scores = gr.Checkbox(True, label="Show Scores")

                    search_btn = gr.Button("🔍 Search Resumes", variant="primary", size="lg")

                with gr.Column(scale=1):
                    metrics_out = gr.Markdown()
                    timing_out = gr.Markdown()

            results_out = gr.HTML()

            # Examples
            gr.Markdown("### 💡 Try These Examples")
            with gr.Row():
                gr.Examples(
                    examples=[
                        ["Senior Software Engineer with expertise in Python, Java, and cloud technologies. Experience with microservices, Docker, Kubernetes."],
                        ["Data Scientist with ML/DL background. Proficient in Python, TensorFlow, PyTorch. Experience with NLP and computer vision."],
                        ["Full Stack Developer with React, Node.js, and MongoDB experience. Strong in responsive design and RESTful APIs."],
                    ],
                    inputs=jd_input,
                    label="Click to load example"
                )

            search_btn.click(
                search_resumes,
                [jd_input, top_k, use_expansion, show_scores],
                [results_out, metrics_out, timing_out]
            )

        # Tab 2: Single Match
        with gr.Tab("📄 Match Single Resume"):
            gr.Markdown("### Calculate match score for a specific resume")

            with gr.Row():
                jd_single = gr.Textbox(
                    label="📝 Job Description",
                    placeholder="Enter job description...",
                    lines=8
                )
                resume_single = gr.Textbox(
                    label="📄 Resume Text",
                    placeholder="Paste resume text here...",
                    lines=8
                )

            with gr.Row():
                show_features = gr.Checkbox(True, label="Show Feature Breakdown")
                match_btn = gr.Button("🎯 Calculate Match", variant="primary", size="lg")

            with gr.Row():
                with gr.Column(scale=2):
                    match_out = gr.HTML()
                with gr.Column(scale=1):
                    features_out = gr.Code(language="json", label="Features")

            match_btn.click(
                match_single_resume,
                [jd_single, resume_single, show_features],
                [match_out, features_out]
            )

        # Tab 3: Info
        with gr.Tab("ℹ️ About"):
            gr.Markdown("""
            ## 📊 System Performance

            ### Evaluation Results (100 paired samples)

            | Metric | Score | Status |
            |--------|-------|--------|
            | P@1 | 94.00% | ✅ Excellent |
            | NDCG@10 | 96.38% | ✅ Excellent |
            | MRR | 95.63% | ✅ Excellent |
            | Hit@1 | 94.00% | ✅ Excellent |

            ### Architecture

            1. **Query Expansion** (Flan-T5-base)
               - Expands queries with related keywords
               - Improves recall by 10-15%

            2. **Hybrid Retrieval** (BM25 + BGE-large)
               - Keyword matching + semantic similarity
               - RRF fusion for optimal results

            3. **Cross-Encoder Reranking** (BGE-reranker-large)
               - Deep query-document interaction
               - Significantly improves precision

            4. **Learning-to-Rank** (LightGBM LambdaRank)
               - 7 features for final ranking
               - Optimized for NDCG@10

            ### Models Used

            - 🔍 **Dense Retrieval:** BAAI/bge-large-en-v1.5
            - 🎯 **Reranker:** BAAI/bge-reranker-large
            - 📝 **Query Expansion:** google/flan-t5-base
            - 🏆 **LTR:** LightGBM LambdaRank

            ### Performance

            - ⚡ **Latency:** ~0.45s per query
            - 🚀 **Throughput:** ~2.2 queries/sec
            - 💾 **GPU Memory:** 8-10 GB
            - 📈 **Scalability:** 10K+ resumes

            ---

            **Built with:** Python, Transformers, FAISS, LightGBM, Gradio
            """)

    return app


# ============================================================================
# LAUNCH
# ============================================================================

def launch_app(pipeline=None, corpus=None, share=False):
    """
    Launch the Gradio app

    Args:
        pipeline: Your pipeline instance (optional, can be set later)
        corpus: Your corpus texts (optional, can be set later)
        share: Whether to create a public link
    """
    if pipeline is not None and corpus is not None:
        set_pipeline(pipeline, corpus)

    app = create_interface()
    app.launch(
        share=share,
        server_name="0.0.0.0",
        server_port=7860,
        show_error=True,
        debug=True
    )


# For Colab usage
if __name__ == "__main__":
    print("""
    ╔══════════════════════════════════════════════════════════════╗
    ║  TYPE 4 Resume Matching - Gradio Interface                  ║
    ╚══════════════════════════════════════════════════════════════╝

    To use this app:

    1. First, run your notebook cells to initialize the pipeline
    2. Then run:

       from gradio_app_standalone import set_pipeline, launch_app
       set_pipeline(pipeline, corpus_texts)
       launch_app(share=True)  # share=True for public link

    Or directly:

       launch_app(pipeline=your_pipeline, corpus=your_corpus, share=True)
    """)


    ╔══════════════════════════════════════════════════════════════╗
    ║  TYPE 4 Resume Matching - Gradio Interface                  ║
    ╚══════════════════════════════════════════════════════════════╝

    To use this app:

    1. First, run your notebook cells to initialize the pipeline
    2. Then run:

       from gradio_app_standalone import set_pipeline, launch_app
       set_pipeline(pipeline, corpus_texts)
       launch_app(share=True)  # share=True for public link

    Or directly:

       launch_app(pipeline=your_pipeline, corpus=your_corpus, share=True)
    


In [ ]:
# Assuming cell -ZQO9W3YAslH has been executed, set_pipeline and launch_app are available.
# Assuming global `run_pipeline`, `resumes` (main corpus), `initialize_bm25`, `initialize_dense` etc. are available from previous cells.

class GradioPipelineAdapter:
    def __init__(self, pipeline_func, original_resumes):
        self._pipeline_func = pipeline_func
        self._original_resumes = original_resumes

    def search(self, query: str, top_k: int = 10, use_query_expansion: bool = True) -> List[Dict]:
        # `run_pipeline` already uses CONFIG.USE_QUERY_EXPANSION globally.
        # It returns a dict, so we extract the 'results' list.
        full_result = self._pipeline_func(query, top_k=top_k)

        # Format the results to match what gradio_app_standalone expects (`text` or `resume` key).
        formatted_results = []
        for r in full_result['results']:
            formatted_results.append({
                'doc_idx': r['doc_idx'],
                'text': r['resume_text'], # Map 'resume_text' to 'text' for Gradio UI
                'resume': r['resume_text'], # Also 'resume'
                'bm25_score': r['bm25_score'],
                'dense_score': r['dense_score'],
                'ce_score': r['ce_score'],
                'final_score': r['final_score']
            })
        return formatted_results

    def build_index(self, corpus_texts_list: List[str]):
        """
        Re-initializes BM25 and Dense components with the given corpus.
        This modifies global state (resumes_text, bm25, dense_encoder, faiss_index).
        This function is called by the Gradio app for single resume matching
        and for restoring the original corpus.
        """
        global resumes_text, bm25, dense_encoder, resume_embeddings, faiss_index, logger

        # Temporarily update the global `resumes_text` which is used by pipeline functions.
        # The Gradio app's `match_single_resume` ensures the original corpus is restored later.
        resumes_text = corpus_texts_list

        logger.info(f"Building/re-building retrieval index with corpus of size: {len(corpus_texts_list)}")
        initialize_bm25(corpus_texts_list)
        initialize_dense(corpus_texts_list) # This re-creates resume_embeddings and faiss_index
        logger.info("Retrieval components (BM25, Dense) re-initialized.")

# Instantiate the adapter with the globally defined `run_pipeline` function and `resumes` corpus.
pipeline_adapter_instance = GradioPipelineAdapter(
    pipeline_func=run_pipeline,
    original_resumes=resumes # `resumes` is the main corpus loaded earlier
)

# Set the pipeline and corpus texts for the Gradio app using the globally defined `resumes`.
# `set_pipeline` and `launch_app` are defined in cell -ZQO9W3YAslH.
set_pipeline(pipeline_adapter_instance, resumes)

# Launch the Gradio app
launch_app(share=True)

✅ Gradio app configured with pipeline!


/tmp/ipykernel_1077/185992136.py:322: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="TYPE 4 Resume Matcher", theme=gr.themes.Soft()) as app:
/tmp/ipykernel_1077/185992136.py:322: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="TYPE 4 Resume Matcher", theme=gr.themes.Soft()) as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://12af6aee3f32b531c2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
